# Анализ комментариев и паттернов

Этот notebook для:
1. Исследования результатов анализа
2. Визуализации паттернов
3. Тестирования генерации комментариев

In [ ]:
import sys
sys.path.append('../src')

import json
import pandas as pd
from pathlib import Path

from analyzer import CommentAnalyzer
from detector import AICommentDetector
from generator import CommentGenerator, GenerationConfig

## 1. Загрузка данных

In [ ]:
# Загрузка последнего анализа
analysis_dir = Path('../data/analysis')
analysis_files = sorted(analysis_dir.glob('analysis_*.json'))

if analysis_files:
    latest_analysis = analysis_files[-1]
    print(f"Загружаем: {latest_analysis.name}")
    
    with open(latest_analysis, 'r', encoding='utf-8') as f:
        analysis_data = json.load(f)
else:
    print("Анализы не найдены. Запустите сначала: python src/main.py")
    analysis_data = None

## 2. Визуализация паттернов

In [ ]:
if analysis_data:
    patterns = analysis_data['aggregate_patterns']
    
    print("=" * 60)
    print("АГРЕГИРОВАННЫЕ ПАТТЕРНЫ")
    print("=" * 60)
    print(f"\nСредняя длина: {patterns['avg_length']:.0f} символов")
    print(f"Среднее кол-во слов: {patterns['avg_word_count']:.0f}")
    print(f"Среднее кол-во предложений: {patterns['avg_sentence_count']:.1f}")
    
    print(f"\nИспользуют параграфы: {patterns['use_paragraphs_pct']:.0f}%")
    print(f"Используют эмодзи: {patterns['use_emoji_pct']:.0f}%")
    print(f"Используют вопросы: {patterns['use_questions_pct']:.0f}%")
    
    print(f"\nНачинают с согласия: {patterns['starts_with_agreement_pct']:.0f}%")
    print(f"Начинают с личного: {patterns['starts_with_personal_pct']:.0f}%")
    
    print(f"\nУпоминают опыт: {patterns['mentions_experience_pct']:.0f}%")
    print(f"Предоставляют ценность: {patterns['provides_value_pct']:.0f}%")
    
    print("\nТОП ключевых фраз:")
    for phrase in patterns.get('common_key_phrases', [])[:5]:
        print(f"  - {phrase}")

## 3. Анализ отдельных комментариев

In [ ]:
if analysis_data:
    # Преобразуем в DataFrame для удобства
    comments_df = pd.DataFrame(analysis_data['individual_analyses'])
    
    print(f"Всего комментариев: {len(comments_df)}")
    print(f"\nСтатистика длины:")
    print(comments_df['length'].describe())
    
    print(f"\nРаспределение:")
    print(f"  С эмодзи: {comments_df['has_emoji'].sum()} ({comments_df['has_emoji'].mean()*100:.0f}%)")
    print(f"  С вопросами: {comments_df['has_question'].sum()} ({comments_df['has_question'].mean()*100:.0f}%)")
    print(f"  Личный опыт: {comments_df['mentions_personal_experience'].sum()} ({comments_df['mentions_personal_experience'].mean()*100:.0f}%)")

## 4. Тестирование детектора AI

In [ ]:
detector = AICommentDetector()

# Тестовые комментарии
test_comments = [
    "It's worth noting that this approach has several advantages. Furthermore, it's important to understand the implications.",
    "Блин, это реально круто! 🔥 У меня тоже был такой опыт...",
    "Согласен! По моему опыту это работает. Попробуйте!"
]

for i, comment in enumerate(test_comments, 1):
    result = detector.detect(comment)
    print(f"\n{i}. {comment[:60]}...")
    print(f"   Тип: {result.comment_type.value}")
    print(f"   Human Score: {result.human_score:.1f}%")
    print(f"   AI Score: {result.ai_score:.1f}%")

## 5. Генерация тестовых комментариев

⚠️ Требуется API ключ в .env файле

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')

api_key = os.getenv('ANTHROPIC_API_KEY') or os.getenv('OPENAI_API_KEY')

if api_key:
    provider = "anthropic" if os.getenv('ANTHROPIC_API_KEY') else "openai"
    generator = CommentGenerator(api_key=api_key, provider=provider)
    
    # Тестовый пост
    test_post = "Сегодня понял важную вещь про тайм-менеджмент: не количество часов, а качество фокуса определяет результат."
    
    config = GenerationConfig(
        post_text=test_post,
        target_length="medium",
        tone="friendly",
        use_emoji=True,
        use_personal_experience=True,
        start_pattern="agreement"
    )
    
    print("Генерация комментария...\n")
    result = generator.generate(config)
    
    print(f"ПОСТ: {test_post}\n")
    print(f"КОММЕНТАРИЙ: {result['comment']}\n")
    print(f"Human Score: {result['human_score']:.1f}%")
    print(f"Попыток: {result['attempts']}")
else:
    print("API ключ не найден. Добавьте в .env файл.")

## 6. Массовая генерация с разными конфигурациями

In [ ]:
if api_key:
    configs = [
        ("Согласие", "agreement"),
        ("Личный опыт", "personal"),
        ("Вопрос", "question"),
        ("Инсайт", "insight")
    ]
    
    for name, pattern in configs:
        config = GenerationConfig(
            post_text=test_post,
            target_length="medium",
            tone="friendly",
            use_emoji=True,
            start_pattern=pattern
        )
        
        result = generator.generate(config, max_attempts=1)
        
        print(f"\n{name}:")
        print(f"  {result['comment']}")
        print(f"  (Human score: {result['human_score']:.1f}%)")